In [1]:
import pandas as pd
df=pd.read_csv("IMDB Dataset.csv")
df.shape


(50000, 2)

In [2]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.shape


(49582, 2)

// CONVERTING TO LOWERCASE


In [6]:
df["review"]=df["review"].str.lower()

//removing the url

In [7]:
import re
def remove_urls(text):
    text=re.sub(r"http\s+","",text)
    return text
df["review"]=df["review"].apply(remove_urls)

// REMOVING PUNCTUATION

In [8]:
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"]=df["review"].apply(remove_punctuations)


 //REMOVE HTML 

In [9]:
def remove_html(text):
    text=re.sub(r"<.* ?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

// REMOVING STOPWORDS

In [10]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shubh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shubh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shubh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords



In [12]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text

df["review"]=df["review"].apply(remove_stopwords)


// STEMMING

In [13]:
from nltk.stem import PorterStemmer
def stemming(text):
    
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return "".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

// ENCODING

In [15]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]
y


0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

// VECTORIZATION

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df["review"])
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5008 stored elements and shape (49582, 5000)>

// CREATE DATASET AND DATALOADER

In [17]:
from sklearn.model_selection import train_test_split 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape

(39665, 5000)

In [18]:
x_test.shape

(9917, 5000)

In [19]:
print(type(x_train))

<class 'scipy.sparse._csr.csr_matrix'>


In [31]:
import torch
from torch.utils.data import TensorDataset,DataLoader


train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)
                        
    

    

// BUILD OUR RNN

In [32]:
import torch.nn as nn
import torch.optim as optim
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
      super().__init__()
      self.hidden_size=hidden_size
      self.num_layers=num_layers
      self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
      self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
      h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
      out,_=self.rnn(x,h0)
      out=self.fc(out[:,-1,:])
      return out
    
    
     
    

In [33]:
 input_size=x_train.shape[1]
 model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())


// TRAINING THE RNN


In [35]:
epochs=10
for epoch in range(epochs):
    model.train()

    for xb,yb in train_loader:
        optimizer.zero_grad()

        xb=xb.unsqueeze(1)
        outputs=model(xb)
        outputs=torch.sigmoid(outputs).squeeze()

        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()
print (f"epoch={epoch+1}/{epochs}and loss={loss.item()}")


epoch=10/10and loss=0.622816264629364


// Evaluation


In [44]:
model.eval()
with torch.no_grad():
    correct_vals=0
    tot_vals=0

    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)

        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals+=yb.size(0)
        correct_vals+=(predicted==yb).sum().item()

    print(f"accuracy={correct_vals/tot_vals*100}")

NameError: name 'model' is not defined